# Epoch/Round Tuning Analysis

This notebook reads the grouped `epoch_round_tuning/tuning-*` runs, summarizes per-round validation MMLU, compares the selected epoch/round pairs across linear FLoRA, nonlinear FLoRA, and nonlinear FFA, and renders Plotly heatmaps for the epoch x round grid.

In [9]:
import os
from pathlib import Path

from IPython.display import display
import pandas as pd
import plotly.express as px

from utils.tuning_analysis import (
    build_extension_requests,
    build_llama_confirmation_requests,
    compare_selected_to_paper,
    compute_epoch_round_selection_metrics,
    load_live_tuning_results,
    load_tuning_results,
    make_tuning_heatmap,
    make_tuning_round_curves,
    select_plateaus,
    summarize_tuning_results,
    write_manifest,
)

CHROME_PATH = Path.home() / '.cache' / 'plotly-chrome' / 'chrome-linux64' / 'chrome'
if CHROME_PATH.exists():
    os.environ.setdefault('BROWSER_PATH', str(CHROME_PATH))

BASE_DIR = Path('.')
NONLINEAR_FLORA_MANIFEST = BASE_DIR / 'tuning_manifests' / 'tinyllama_nonlinear_flora_stratified_seed0_r10.tsv'
CUMULATIVE_NONLINEAR_FLORA_RUN_ROOT = BASE_DIR / 'epoch_round_tuning_nonlinear_flora_cumulative'
LIVE_LOG_DIR = BASE_DIR / 'logs'
TUNING_RUN_ROOTS = [
    BASE_DIR / 'epoch_round_tuning',
    BASE_DIR / 'epoch_round_tuning_r10',
    BASE_DIR / 'epoch_round_tuning_dolly_dirichlet_r10',
    BASE_DIR / 'exp2-tinyllama-nonlinear-wiz',
    BASE_DIR / 'exp2-tinyllama-nonlinear-3round-heter-wiz',
    BASE_DIR / 'epoch_round_tuning_wiz_stratified_r10',
    # BASE_DIR / 'exp2-llama-nonlinear-1round-homo-wiz',
    # BASE_DIR / 'exp2-llama-nonlinear-1round-heter-wiz',
]
CENTRALIZED_LORA_RUNS = [
    {
        'dataset': 'dolly_stratified',
        'model': 'tinyllama',
        'epochs': 3,
        'metrics_path': Path('/home/trieu.vy.tran/LoRA/examples/NLU/centralized_instruction_qa/full/dolly_tinyllama_seed0/metrics.json'),
    },
    {
        'dataset': 'wiz_stratified',
        'model': 'tinyllama',
        'epochs': 3,
        'metrics_path': Path('/home/trieu.vy.tran/LoRA/examples/NLU/centralized_instruction_qa/full/wizard_tinyllama_seed0/metrics.json'),
    },
]
OUTPUT_DIR = BASE_DIR / 'tuning_analysis_outputs'
FIGURE_DIR = OUTPUT_DIR / 'epoch_round_heatmaps'
ROUND_CURVE_DIR = BASE_DIR / 'plots_epoch_round_tuning'

OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
ROUND_CURVE_DIR.mkdir(exist_ok=True)


## Load And Select

In [10]:
import importlib
import utils.tuning_analysis as tuning_analysis

tuning_analysis = importlib.reload(tuning_analysis)
load_live_tuning_results = tuning_analysis.load_live_tuning_results
load_tuning_results = tuning_analysis.load_tuning_results
summarize_tuning_results = tuning_analysis.summarize_tuning_results
select_plateaus = tuning_analysis.select_plateaus
compute_epoch_round_selection_metrics = tuning_analysis.compute_epoch_round_selection_metrics
make_tuning_heatmap = tuning_analysis.make_tuning_heatmap


def _manifest_model_tag(model):
    return 'llama' if model in {'llama', 'llama-7b'} else str(model)


def _manifest_log_path(row, run_root):
    method = str(row['method'])
    model = _manifest_model_tag(str(row['model']))
    epochs = int(row['epochs'])
    rounds = int(row['rounds'])
    seed = int(row['seed'])
    run_dir = run_root / f"tuning-{method}-{row['dataset']}-{model}-{row['setting']}-e{epochs}-r{rounds}" / f"seed{seed}"
    if method == 'flora':
        return run_dir / '10log.txt'
    return run_dir / '10' / 'log.txt'


def _live_observed_rounds(live_scores, row):
    if live_scores is None or live_scores.empty:
        return 0
    model = 'llama' if str(row['model']) in {'llama', 'llama-7b'} else str(row['model'])
    matches = live_scores[
        (live_scores['Method key'] == str(row['method']))
        & (live_scores['Dataset'] == str(row['dataset']))
        & (live_scores['Model key'] == model)
        & (live_scores['Setting key'] == str(row['setting']))
        & (live_scores['Local epochs'].astype(int) == int(row['epochs']))
        & (live_scores['Seed'].astype(int) == int(row['seed']))
    ]
    if matches.empty:
        return 0
    return int(matches['Observed rounds'].max())


def build_manifest_status(manifest_path, run_root, live_scores=None):
    if not manifest_path.exists():
        return pd.DataFrame()

    status_rows = []
    manifest = pd.read_csv(manifest_path, sep='	')
    for _, row in manifest.iterrows():
        log_path = _manifest_log_path(row, run_root)
        observed_rounds = 0
        result_source = 'none'
        if log_path.exists():
            observed_rounds = sum(1 for line in log_path.read_text().splitlines() if line.strip())
            result_source = 'score log'
        else:
            observed_rounds = _live_observed_rounds(live_scores, row)
            if observed_rounds:
                result_source = 'slurm stdout'
        config_rounds = int(row['rounds'])
        complete_run = observed_rounds >= config_rounds and result_source == 'score log'
        status = 'complete' if complete_run else ('running/partial' if observed_rounds else 'missing')
        status_rows.append(
            {
                'Method': row['method'],
                'Dataset': row['dataset'],
                'Model': row['model'],
                'Setting': row['setting'],
                'Local epochs': int(row['epochs']),
                'Config rounds': config_rounds,
                'Seed': int(row['seed']),
                'Observed rounds': observed_rounds,
                'Complete run': complete_run,
                'Status': status,
                'Result source': result_source,
                'Log path': str(log_path),
            }
        )
    return pd.DataFrame(status_rows)


live_scores = load_live_tuning_results(
    LIVE_LOG_DIR,
    run_roots=CUMULATIVE_NONLINEAR_FLORA_RUN_ROOT,
)
live_preview_scores = live_scores.copy()
if not live_preview_scores.empty:
    live_preview_scores['Method key'] = 'nonlinear_flora'
    live_preview_scores['Method'] = 'Nonlinear FLoRA'
    live_preview_scores['Run status'] = 'Live partial'

cumulative_nonlinear_flora_status = build_manifest_status(
    NONLINEAR_FLORA_MANIFEST,
    CUMULATIVE_NONLINEAR_FLORA_RUN_ROOT,
    live_scores=live_scores,
)

if cumulative_nonlinear_flora_status.empty:
    print(f'No cumulative nonlinear FLoRA live status found under {CUMULATIVE_NONLINEAR_FLORA_RUN_ROOT}.')
else:
    partial_count = int((cumulative_nonlinear_flora_status['Observed rounds'] > 0).sum())
    print(f'Cumulative nonlinear FLoRA live/complete rows with scores: {partial_count}/{len(cumulative_nonlinear_flora_status)}.')
    display(
        cumulative_nonlinear_flora_status
        .drop(columns=['Log path'])
        .sort_values(['Dataset', 'Setting', 'Local epochs'])
        .reset_index(drop=True)
    )

# Complete-run tables remain the source of truth for selected epochs, plateaus, and CSVs.
# Drop the old nonlinear FLoRA overwrite-behavior runs; cumulative nonlinear FLoRA is shown via live preview.
all_complete_scores = load_tuning_results(TUNING_RUN_ROOTS, complete_only=True)
obsolete_nonlinear_mask = all_complete_scores['Method key'].eq('nonlinear_flora')
obsolete_nonlinear_run_count = all_complete_scores.loc[obsolete_nonlinear_mask, 'Run dir'].nunique()
tuning_scores = all_complete_scores.loc[~obsolete_nonlinear_mask].copy()
tuning_summary = summarize_tuning_results(tuning_scores)
tuning_plateaus, tuning_selected = select_plateaus(tuning_summary)
curve_diagnostics, optimal_epoch_round_pairs, marginal_round_gains = compute_epoch_round_selection_metrics(
    tuning_summary,
    plateaus=tuning_plateaus,
    selected_pairs=tuning_selected,
    tolerance=1.0,
)

# Plot-only preview data includes live cumulative nonlinear FLoRA rows from Slurm stdout.
plot_scores = pd.concat([tuning_scores, live_preview_scores], ignore_index=True, sort=False)
plot_summary = summarize_tuning_results(plot_scores)

run_count = tuning_scores['Run dir'].nunique() if not tuning_scores.empty else 0
case_count = tuning_selected.shape[0] if not tuning_selected.empty else 0
live_run_count = live_preview_scores['Run dir'].nunique() if not live_preview_scores.empty else 0
print(f'Loaded {len(tuning_scores)} per-round scores from {run_count} complete runs under {TUNING_RUN_ROOTS}.')
print(f'Excluded {obsolete_nonlinear_run_count} old nonlinear FLoRA complete runs with overwrite behavior.')
print(f'Loaded {len(live_preview_scores)} live preview scores from {live_run_count} cumulative nonlinear FLoRA runs as Nonlinear FLoRA.')
print(f'Selected {case_count} method/dataset/model/setting configurations from complete runs only.')

display(tuning_selected.round(2))
display(tuning_plateaus.round(2))


Cumulative nonlinear FLoRA live/complete rows with scores: 16/16.


,Method,Dataset,Model,Setting,Local epochs,Config rounds,Seed,Observed rounds,Complete run,Status,Result source
0,nonlinear_flora,dolly_stratified,tinyllama,heter,1,10,0,10,True,complete,score log
1,nonlinear_flora,dolly_stratified,tinyllama,heter,2,10,0,10,True,complete,score log
2,nonlinear_flora,dolly_stratified,tinyllama,heter,3,10,0,10,True,complete,score log
3,nonlinear_flora,dolly_stratified,tinyllama,heter,5,10,0,10,True,complete,score log
4,nonlinear_flora,dolly_stratified,tinyllama,homo,1,10,0,10,True,complete,score log
5,nonlinear_flora,dolly_stratified,tinyllama,homo,2,10,0,10,True,complete,score log
6,nonlinear_flora,dolly_stratified,tinyllama,homo,3,10,0,10,True,complete,score log
7,nonlinear_flora,dolly_stratified,tinyllama,homo,5,10,0,10,True,complete,score log
8,nonlinear_flora,wiz_stratified,tinyllama,heter,1,10,0,10,True,complete,score log
9,nonlinear_flora,wiz_stratified,tinyllama,heter,2,10,0,10,True,complete,score log


Loaded 740 per-round scores from 74 complete runs under [PosixPath('epoch_round_tuning'), PosixPath('epoch_round_tuning_r10'), PosixPath('epoch_round_tuning_dolly_dirichlet_r10'), PosixPath('exp2-tinyllama-nonlinear-wiz'), PosixPath('exp2-tinyllama-nonlinear-3round-heter-wiz'), PosixPath('epoch_round_tuning_wiz_stratified_r10')].
Excluded 17 old nonlinear FLoRA complete runs with overwrite behavior.
Loaded 242 live preview scores from 27 cumulative nonlinear FLoRA runs as Nonlinear FLoRA.
Selected 18 method/dataset/model/setting configurations from complete runs only.


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Selected epochs,Selected round,Selected accuracy,Selected std,Selected seed count,Selected seeds,Selected compute cost,Best epochs,Best round,Best accuracy,Accuracy gap to best,Low-cost tie count
0,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,3,4,28.52,0.0,1,0,12,5,9,29.05,0.52,1
1,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,2,3,29.90,0.0,1,0,6,5,7,30.87,0.97,1
4,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,heter,Heter,3,5,29.62,0.0,1,0,15,3,5,29.62,0.00,1
5,ffa,Nonlinear FFA,dolly_stratified,Dolly stratified,tinyllama,TinyLlama,homo,Homo,3,7,30.62,0.0,1,0,21,3,10,31.26,0.65,1
10,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,heter,Heter,1,6,33.19,0.0,1,0,6,1,10,33.63,0.44,1
11,ffa,Nonlinear FFA,wiz,Wizard,tinyllama,TinyLlama,homo,Homo,2,1,30.69,0.0,1,0,2,5,4,31.55,0.86,2
14,ffa,Nonlinear FFA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,heter,Heter,2,3,32.46,0.0,1,0,6,5,3,33.37,0.91,1
15,ffa,Nonlinear FFA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,homo,Homo,2,2,31.81,0.0,1,0,4,5,1,32.35,0.54,1
2,flora,Linear FLoRA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,2,1,29.38,0.0,1,0,2,2,5,29.96,0.58,1
3,flora,Linear FLoRA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,3,2,31.03,0.0,1,0,6,3,2,31.03,0.00,1


,Method key,Method,Dataset,Dataset label,Model key,Model,Setting key,Setting,Local epochs,Plateau round,Plateau accuracy,Best round,Best accuracy,Max round observed,Declined after best
0,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,1,8,26.33,10,27.24,10,False
1,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,2,7,27.52,9,28.38,10,False
2,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,3,4,28.52,6,28.99,10,True
3,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,heter,Heter,5,4,28.88,9,29.05,10,True
4,ffa,Nonlinear FFA,dolly,Dolly,tinyllama,TinyLlama,homo,Homo,1,6,29.29,6,29.29,10,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,flora,Linear FLoRA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,heter,Heter,5,1,42.25,1,42.25,10,True
70,flora,Linear FLoRA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,homo,Homo,1,1,46.74,1,46.74,10,True
71,flora,Linear FLoRA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,homo,Homo,2,1,42.06,1,42.06,10,True
72,flora,Linear FLoRA,wiz_stratified,Wizard stratified,tinyllama,TinyLlama,homo,Homo,3,1,38.97,2,38.97,10,True


In [11]:
tuning_scores.to_csv(OUTPUT_DIR / 'tuning_scores.csv', index=False)
tuning_summary.to_csv(OUTPUT_DIR / 'tuning_summary.csv', index=False)
tuning_plateaus.to_csv(OUTPUT_DIR / 'tuning_plateaus.csv', index=False)
tuning_selected.to_csv(OUTPUT_DIR / 'tuning_selected.csv', index=False)
curve_diagnostics.to_csv(OUTPUT_DIR / 'epoch_round_curve_diagnostics.csv', index=False)
optimal_epoch_round_pairs.to_csv(OUTPUT_DIR / 'epoch_round_optimal_pairs.csv', index=False)
marginal_round_gains.to_csv(OUTPUT_DIR / 'epoch_round_marginal_gains.csv', index=False)
print(f'Wrote CSV summaries to {OUTPUT_DIR}.')


Wrote CSV summaries to tuning_analysis_outputs.


## Optimal Epoch/Round Metrics

In [12]:
OPTIMAL_DISPLAY_COLUMNS = [
    'Method',
    'Dataset label',
    'Setting',
    'Selected epochs',
    'Selected round',
    'Selected accuracy',
    'Selected compute cost',
    'Selected post-peak degradation',
    'Best epochs',
    'Best round',
    'Best accuracy',
    'Accuracy gap to best',
    'Communication-efficient epochs',
    'Communication-efficient round',
    'Communication-efficient accuracy',
    'Communication-efficient compute cost',
    'Selection matches tuning_selected',
]
DIAGNOSTIC_DISPLAY_COLUMNS = [
    'Method',
    'Dataset label',
    'Setting',
    'Local epochs',
    'Plateau round',
    'Plateau accuracy',
    'Best round',
    'Best accuracy',
    'Final round',
    'Final accuracy',
    'Post-peak degradation',
    'Worst later degradation',
    'Later rounds hurt > tolerance',
]
MARGINAL_SUMMARY_COLUMNS = [
    'Method',
    'Dataset label',
    'Setting',
    'Local epochs',
    'Mean marginal gain',
    'Last marginal gain',
    'Negative marginal rounds',
]


def _method_rows(frame, method_key):
    if frame.empty:
        return frame
    return frame[frame['Method key'].eq(method_key)].copy()


def _display_optimal_metrics(method_key, label):
    optimal_rows = _method_rows(optimal_epoch_round_pairs, method_key)
    diagnostic_rows = _method_rows(curve_diagnostics, method_key)
    marginal_rows = _method_rows(marginal_round_gains, method_key)

    print(f'{label}: efficient optimal epoch/round pairs within 1.0 percentage point of best')
    if optimal_rows.empty:
        print(f'No complete-run {label} rows available.')
    else:
        display(
            optimal_rows[OPTIMAL_DISPLAY_COLUMNS]
            .sort_values(['Dataset label', 'Setting'])
            .reset_index(drop=True)
            .round(2)
        )

    print(f'{label}: per-epoch plateau and degradation diagnostics')
    if diagnostic_rows.empty:
        print(f'No complete-run {label} diagnostics available.')
    else:
        display(
            diagnostic_rows[DIAGNOSTIC_DISPLAY_COLUMNS]
            .sort_values(['Dataset label', 'Setting', 'Local epochs'])
            .reset_index(drop=True)
            .round(2)
        )

    print(f'{label}: marginal round-gain summary')
    if marginal_rows.empty:
        print(f'No complete-run {label} marginal gains available.')
    else:
        marginal_summary = (
            marginal_rows
            .sort_values(['Dataset label', 'Setting', 'Local epochs', 'Round'])
            .groupby(['Method', 'Dataset label', 'Setting', 'Local epochs'], as_index=False)
            .agg(
                **{
                    'Mean marginal gain': ('Marginal accuracy gain', 'mean'),
                    'Last marginal gain': ('Marginal accuracy gain', 'last'),
                    'Negative marginal rounds': ('Marginal accuracy gain', lambda values: int((values < 0).sum())),
                }
            )
        )
        display(marginal_summary[MARGINAL_SUMMARY_COLUMNS].round(2))


print('Optimal metrics are computed from complete runs only; live/partial preview rows stay out of these selections.')
_display_optimal_metrics('flora', 'Linear FLoRA')

if optimal_epoch_round_pairs.empty:
    print('No reusable all-method optimal table available.')
else:
    print('All complete-run methods: reusable optimal-pair table')
    display(
        optimal_epoch_round_pairs[OPTIMAL_DISPLAY_COLUMNS]
        .sort_values(['Dataset label', 'Setting', 'Method'])
        .reset_index(drop=True)
        .round(2)
    )


Optimal metrics are computed from complete runs only; live/partial preview rows stay out of these selections.
Linear FLoRA: efficient optimal epoch/round pairs within 1.0 percentage point of best


,Method,Dataset label,Setting,Selected epochs,Selected round,Selected accuracy,Selected compute cost,Selected post-peak degradation,Best epochs,Best round,Best accuracy,Accuracy gap to best,Communication-efficient epochs,Communication-efficient round,Communication-efficient accuracy,Communication-efficient compute cost,Selection matches tuning_selected
0,Linear FLoRA,Dolly,Heter,2,1,29.38,2,1.12,2,5,29.96,0.58,2,1,29.38,2,True
1,Linear FLoRA,Dolly,Homo,3,2,31.03,6,2.78,3,2,31.03,0.00,3,2,31.03,6,True
2,Linear FLoRA,Dolly stratified,Heter,5,1,28.34,5,3.25,2,8,29.11,0.77,5,1,28.34,5,True
3,Linear FLoRA,Dolly stratified,Homo,1,5,28.63,5,0.67,5,4,29.61,0.98,3,2,28.85,6,True
4,Linear FLoRA,RTE stratified,Heter,8,10,60.65,80,0.00,8,10,60.65,0.00,8,10,60.65,80,True
5,Linear FLoRA,RTE stratified,Homo,5,9,61.37,45,0.00,5,10,62.09,0.72,8,8,61.73,64,True
6,Linear FLoRA,Wizard,Heter,3,1,46.22,3,7.88,3,1,46.22,0.00,3,1,46.22,3,True
7,Linear FLoRA,Wizard,Homo,1,1,45.10,1,9.52,1,1,45.10,0.00,1,1,45.10,1,True
8,Linear FLoRA,Wizard stratified,Heter,3,1,45.74,3,9.92,3,1,45.74,0.00,3,1,45.74,3,True
9,Linear FLoRA,Wizard stratified,Homo,1,1,46.74,1,11.40,1,1,46.74,0.00,1,1,46.74,1,True


Linear FLoRA: per-epoch plateau and degradation diagnostics


,Method,Dataset label,Setting,Local epochs,Plateau round,Plateau accuracy,Best round,Best accuracy,Final round,Final accuracy,Post-peak degradation,Worst later degradation,Later rounds hurt > tolerance
0,Linear FLoRA,Dolly,Heter,1,2,27.64,8,28.37,10,27.14,1.23,1.23,True
1,Linear FLoRA,Dolly,Heter,2,1,29.38,5,29.96,10,28.83,1.12,2.32,True
2,Linear FLoRA,Dolly,Heter,3,2,29.24,3,29.42,10,27.60,1.82,2.59,True
3,Linear FLoRA,Dolly,Heter,5,1,29.09,2,29.24,10,26.17,3.07,3.10,True
4,Linear FLoRA,Dolly,Homo,1,10,29.86,10,29.86,10,29.86,0.00,0.00,False
5,Linear FLoRA,Dolly,Homo,2,5,30.54,5,30.54,10,29.55,0.98,1.33,True
6,Linear FLoRA,Dolly,Homo,3,2,31.03,2,31.03,10,28.25,2.78,3.44,True
7,Linear FLoRA,Dolly,Homo,5,2,28.85,8,29.60,10,28.20,1.40,2.20,True
8,Linear FLoRA,Dolly stratified,Heter,1,5,28.03,10,28.87,10,28.87,0.00,0.00,False
9,Linear FLoRA,Dolly stratified,Heter,2,3,28.16,8,29.11,10,26.86,2.24,2.42,True


Linear FLoRA: marginal round-gain summary


,Method,Dataset label,Setting,Local epochs,Mean marginal gain,Last marginal gain,Negative marginal rounds
0,Linear FLoRA,Dolly,Heter,1,0.31,-0.81,5
1,Linear FLoRA,Dolly,Heter,2,-0.06,1.20,4
2,Linear FLoRA,Dolly,Heter,3,-0.09,-0.52,5
3,Linear FLoRA,Dolly,Heter,5,-0.32,-1.11,4
4,Linear FLoRA,Dolly,Homo,1,1.32,1.61,4
5,Linear FLoRA,Dolly,Homo,2,0.17,0.05,3
6,Linear FLoRA,Dolly,Homo,3,0.19,-0.25,5
7,Linear FLoRA,Dolly,Homo,5,0.09,0.80,4
8,Linear FLoRA,Dolly stratified,Heter,1,0.80,2.03,4
9,Linear FLoRA,Dolly stratified,Heter,2,-0.00,0.17,3


All complete-run methods: reusable optimal-pair table


,Method,Dataset label,Setting,Selected epochs,Selected round,Selected accuracy,Selected compute cost,Selected post-peak degradation,Best epochs,Best round,Best accuracy,Accuracy gap to best,Communication-efficient epochs,Communication-efficient round,Communication-efficient accuracy,Communication-efficient compute cost,Selection matches tuning_selected
0,Linear FLoRA,Dolly,Heter,2,1,29.38,2,1.12,2,5,29.96,0.58,2,1,29.38,2,True
1,Nonlinear FFA,Dolly,Heter,3,4,28.52,12,1.59,5,9,29.05,0.52,3,4,28.52,12,True
2,Linear FLoRA,Dolly,Homo,3,2,31.03,6,2.78,3,2,31.03,0.00,3,2,31.03,6,True
3,Nonlinear FFA,Dolly,Homo,2,3,29.90,6,0.77,5,7,30.87,0.97,2,3,29.90,6,True
4,Linear FLoRA,Dolly stratified,Heter,5,1,28.34,5,3.25,2,8,29.11,0.77,5,1,28.34,5,True
5,Nonlinear FFA,Dolly stratified,Heter,3,5,29.62,15,1.67,3,5,29.62,0.00,3,5,29.62,15,True
6,Linear FLoRA,Dolly stratified,Homo,1,5,28.63,5,0.67,5,4,29.61,0.98,3,2,28.85,6,True
7,Nonlinear FFA,Dolly stratified,Homo,3,7,30.62,21,0.00,3,10,31.26,0.65,3,7,30.62,21,True
8,Linear FLoRA,RTE stratified,Heter,8,10,60.65,80,0.00,8,10,60.65,0.00,8,10,60.65,80,True
9,Linear FLoRA,RTE stratified,Homo,5,9,61.37,45,0.00,5,10,62.09,0.72,8,8,61.73,64,True


## Method Alignment

In [13]:
if tuning_selected.empty:
    print('No selected configurations yet.')
else:
    index_columns = ['Dataset label', 'Model', 'Setting']
    alignment = tuning_selected.pivot(
        index=index_columns,
        columns='Method',
        values=['Selected epochs', 'Selected round', 'Selected accuracy', 'Best epochs', 'Best round', 'Best accuracy'],
    )
    display(alignment.round(2))

    winners = (
        tuning_selected.sort_values('Selected accuracy', ascending=False)
        .groupby(index_columns, as_index=False)
        .first()[index_columns + ['Method', 'Selected epochs', 'Selected round', 'Selected accuracy', 'Best accuracy']]
    )
    display(winners.round(2))

Selected epochs                \
Method                                    Linear FLoRA Nonlinear FFA   
Dataset label     Model        Setting                                 
Dolly             TinyLlama    Heter               2.0           3.0   
                               Homo                3.0           2.0   
Dolly stratified  TinyLlama    Heter               5.0           3.0   
                               Homo                1.0           3.0   
RTE stratified    RoBERTa-base Heter               8.0           NaN   
                               Homo                5.0           NaN   
Wizard            TinyLlama    Heter               3.0           1.0   
                               Homo                1.0           2.0   
Wizard stratified TinyLlama    Heter               3.0           2.0   
                               Homo                1.0           2.0   

                                       Selected round                \
Method                                   Linear FLoRA Nonlinear FFA   
Dataset label     Model        Setting                                
Dolly             TinyLlama    Heter              1.0           4.0   
                               Homo               2.0           3.0   
Dolly stratified  TinyLlama    Heter              1.0           5.0   
                               Homo               5.0           7.0   
RTE stratified    RoBERTa-base Heter             10.0           NaN   
                               Homo               9.0           NaN   
Wizard            TinyLlama    Heter              1.0           6.0   
                               Homo               1.0           1.0   
Wizard stratified TinyLlama    Heter              1.0           3.0   
                               Homo               1.0           2.0   

                                       Selected accuracy                \
Method                                      Linear FLoRA Nonlinear FFA   
Dataset label     Model        Setting                                   
Dolly             TinyLlama    Heter               29.38         28.52   
                               Homo                31.03         29.90   
Dolly stratified  TinyLlama    Heter               28.34         29.62   
                               Homo                28.63         30.62   
RTE stratified    RoBERTa-base Heter               60.65           NaN   
                               Homo                61.37           NaN   
Wizard            TinyLlama    Heter               46.22         33.19   
                               Homo                45.10         30.69   
Wizard stratified TinyLlama    Heter               45.74         32.46   
                               Homo                46.74         31.81   

                                        Best epochs                \
Method                                 Linear FLoRA Nonlinear FFA   
Dataset label     Model        Setting                              
Dolly             TinyLlama    Heter            2.0           5.0   
                               Homo             3.0           5.0   
Dolly stratified  TinyLlama    Heter            2.0           3.0   
                               Homo             5.0           3.0   
RTE stratified    RoBERTa-base Heter            8.0           NaN   
                               Homo             5.0           NaN   
Wizard            TinyLlama    Heter            3.0           1.0   
                               Homo             1.0           5.0   
Wizard stratified TinyLlama    Heter            3.0           5.0   
                               Homo             1.0           5.0   

                                         Best round                \
Method                                 Linear FLoRA Nonlinear FFA   
Dataset label     Model        Setting                              
Dolly             TinyLlama    Heter            5.0           9.0   
                               Homo      

,Dataset label,Model,Setting,Method,Selected epochs,Selected round,Selected accuracy,Best accuracy
0,Dolly,TinyLlama,Heter,Linear FLoRA,2,1,29.38,29.96
1,Dolly,TinyLlama,Homo,Linear FLoRA,3,2,31.03,31.03
2,Dolly stratified,TinyLlama,Heter,Nonlinear FFA,3,5,29.62,29.62
3,Dolly stratified,TinyLlama,Homo,Nonlinear FFA,3,7,30.62,31.26
4,RTE stratified,RoBERTa-base,Heter,Linear FLoRA,8,10,60.65,60.65
5,RTE stratified,RoBERTa-base,Homo,Linear FLoRA,5,9,61.37,62.09
6,Wizard,TinyLlama,Heter,Linear FLoRA,3,1,46.22,46.22
7,Wizard,TinyLlama,Homo,Linear FLoRA,1,1,45.10,45.10
8,Wizard stratified,TinyLlama,Heter,Linear FLoRA,3,1,45.74,45.74
9,Wizard stratified,TinyLlama,Homo,Linear FLoRA,1,1,46.74,46.74


## Paper Baseline Comparison

The Wizard and Dolly paper baselines are taken from `flora_results_analysis.ipynb`. Paper results are plotted only at their reported communication round, not as horizontal lines across all rounds. Dolly tuning here uses the stratified split, so the Dolly paper point is contextual rather than an exact split-matched baseline.

In [14]:
# paper_comparison = compare_selected_to_paper(tuning_selected)
# if paper_comparison.empty:
#     print('No paper baselines matched the selected tuning rows.')
# else:
#     paper_comparison.to_csv(OUTPUT_DIR / 'tuning_vs_paper.csv', index=False)
#     display(paper_comparison.round(2))


## Round Curves

The fixed-epoch method comparison plots answer the direct question: at local epoch 1, 2, 3, or 5, which method has the better round curve? The all-curve overview plots stay separate so the main comparison stays readable. I would avoid overlaying Dolly and Dolly stratified in this same view because it doubles the curve count and makes the method comparison harder to read; a split-focused view should be faceted by split if we want to inspect that question.


In [15]:
METHOD_ORDER = ['Linear FLoRA', 'Nonlinear FLoRA', 'Nonlinear FFA']
SETTING_ORDER = ['Homo', 'Heter']
RUN_STATUS_ORDER = ['Complete', 'Partial log', 'Live partial']
RUN_STATUS_LINE_DASH_MAP = {'Complete': 'solid', 'Partial log': 'dot', 'Live partial': 'dash'}
RUN_STATUS_SYMBOL_MAP = {'Complete': 'circle', 'Partial log': 'x', 'Live partial': 'diamond'}
METHOD_LINE_DASH_MAP = {
    'Linear FLoRA': 'solid',
    'Nonlinear FLoRA': 'solid',
    'Nonlinear FFA': 'dot',
}


def _slug(value):
    return str(value).lower().replace(' ', '_').replace('-', '_').replace('/', '_')


def _sorted_unique_ints(values):
    return sorted({int(value) for value in values})


def _category_orders(data):
    orders = {
        'Method': [method for method in METHOD_ORDER if method in set(data['Method'])],
        'Setting': [setting for setting in SETTING_ORDER if setting in set(data['Setting'])],
        'Local epochs': [str(epoch) for epoch in _sorted_unique_ints(data['Local epochs'])],
    }
    if 'Run status' in data.columns:
        orders['Run status'] = [status for status in RUN_STATUS_ORDER if status in set(data['Run status'])]
    return orders


def _format_facet_labels(fig):
    fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))


def _clean_trace_names(fig):
    suffixes = [f', {status}' for status in RUN_STATUS_ORDER]
    for trace in fig.data:
        for suffix in suffixes:
            if trace.name.endswith(suffix):
                trace.name = trace.name[:-len(suffix)]
                break


def _save_plot(fig, output_stem, png_warnings):
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    png_path = output_stem.with_suffix('.png')
    try:
        fig.write_image(png_path, scale=2)
    except Exception as exc:
        png_path = None
        message = str(exc).strip() or type(exc).__name__
        png_warnings.append(message.splitlines()[0])
    return png_path


def make_epoch_method_comparison(summary, dataset, epoch):
    data = summary[
        (summary['Dataset'] == dataset)
        & (summary['Local epochs'].astype(int) == int(epoch))
    ].copy()
    if data.empty:
        return None

    data = data.sort_values(['Model', 'Setting', 'Method', 'Round'])
    dataset_label = data['Dataset label'].iloc[0]
    model_label = ', '.join(sorted(data['Model'].unique()))
    facet_row = 'Model' if data['Model key'].nunique() > 1 else None
    use_run_status = 'Run status' in data.columns
    hover_data = {
        'Local epochs': True,
        'Std accuracy': ':.2f',
        'Seed count': True,
    }
    if use_run_status:
        hover_data['Run status'] = True
    if 'Result source' in data.columns:
        hover_data['Result source'] = True

    fig = px.line(
        data,
        x='Round',
        y='Mean accuracy',
        color='Method',
        line_dash='Run status' if use_run_status else None,
        symbol='Run status' if use_run_status else None,
        facet_col='Setting',
        facet_row=facet_row,
        markers=True,
        template='plotly_white',
        title=f'{dataset_label} {model_label} epoch {epoch}: method comparison',
        labels={'Mean accuracy': 'MMLU accuracy (%)'},
        category_orders=_category_orders(data),
        line_dash_map=RUN_STATUS_LINE_DASH_MAP if use_run_status else None,
        symbol_map=RUN_STATUS_SYMBOL_MAP if use_run_status else None,
        hover_data=hover_data,
    )
    fig.update_layout(width=1000, height=400, legend_title_text='')
    fig.update_xaxes(dtick=1, showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    fig.update_yaxes(showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    _clean_trace_names(fig)
    _format_facet_labels(fig)
    return fig


def make_dataset_all_curves(summary, dataset):
    data = summary[summary['Dataset'] == dataset].copy()
    if data.empty:
        return None

    data['Local epochs'] = data['Local epochs'].astype(int).astype(str)
    data = data.sort_values(['Model', 'Setting', 'Method', 'Local epochs', 'Round'])
    dataset_label = data['Dataset label'].iloc[0]
    facet_row = 'Model' if data['Model key'].nunique() > 1 else None
    use_run_status = 'Run status' in data.columns
    hover_data = {
        'Std accuracy': ':.2f',
        'Seed count': True,
        'Method': True,
    }
    if use_run_status:
        hover_data['Run status'] = True
    if 'Result source' in data.columns:
        hover_data['Result source'] = True

    fig = px.line(
        data,
        x='Round',
        y='Mean accuracy',
        color='Local epochs',
        line_dash='Method',
        symbol='Run status' if use_run_status else None,
        facet_col='Setting',
        facet_row=facet_row,
        markers=True,
        template='plotly_white',
        title=f'{dataset_label}: all tuning round curves',
        labels={'Mean accuracy': 'MMLU accuracy (%)', 'Local epochs': 'Epochs'},
        category_orders=_category_orders(data),
        line_dash_map=METHOD_LINE_DASH_MAP,
        symbol_map=RUN_STATUS_SYMBOL_MAP if use_run_status else None,
        hover_data=hover_data,
    )
    fig.update_layout(width=1000, height=400, legend_title_text='')
    fig.update_xaxes(dtick=1, showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    fig.update_yaxes(showgrid=True, gridcolor='#d9d9d9', gridwidth=1)
    _format_facet_labels(fig)
    return fig


plot_summary_for_curves = plot_summary if 'plot_summary' in globals() else tuning_summary
if plot_summary_for_curves.empty:
    print('No tuning summary to plot.')
else:
    png_warnings = []
    saved_png = []
    datasets = plot_summary_for_curves[['Dataset', 'Dataset label']].drop_duplicates().sort_values('Dataset label')
    epochs = _sorted_unique_ints(plot_summary_for_curves['Local epochs'])

    comparison_root = ROUND_CURVE_DIR / 'fixed_epoch_method_comparisons'
    overview_root = ROUND_CURVE_DIR / 'all_curves'

    print('Fixed-epoch method comparisons')
    for _, dataset_row in datasets.iterrows():
        dataset = dataset_row['Dataset']
        dataset_label = dataset_row['Dataset label']
        dataset_dir = comparison_root / _slug(dataset_label)
        for epoch in epochs:
            fig = make_epoch_method_comparison(plot_summary_for_curves, dataset, epoch)
            if fig is None:
                print(f'No data for {dataset_label} epoch {epoch}; skipping.')
                continue
            png_path = _save_plot(fig, dataset_dir / f'epoch_{epoch}', png_warnings)
            if png_path is not None:
                saved_png.append(png_path)
            print(f'{dataset_label} epoch {epoch}')
            fig.show()

    print('All-curve dataset overviews')
    for _, dataset_row in datasets.iterrows():
        dataset = dataset_row['Dataset']
        dataset_label = dataset_row['Dataset label']
        fig = make_dataset_all_curves(plot_summary_for_curves, dataset)
        if fig is None:
            continue
        png_path = _save_plot(fig, overview_root / _slug(dataset_label), png_warnings)
        if png_path is not None:
            saved_png.append(png_path)
        print(dataset_label)
        fig.show()

    if saved_png:
        print(f'Wrote {len(saved_png)} PNG files under {ROUND_CURVE_DIR}.')
    if png_warnings:
        print('PNG export skipped or incomplete. Install/enable kaleido for static PNG export.')
        print(f'First PNG export warning: {png_warnings[0]}')


Fixed-epoch method comparisons
Dolly epoch 1


Dolly epoch 2


Dolly epoch 3


Dolly epoch 5


No data for Dolly epoch 8; skipping.
Dolly stratified epoch 1


...

Dolly stratified


RTE stratified


Wizard


Wizard stratified


PNG export skipped or incomplete. Install/enable kaleido for static PNG export.
First PNG export warning: Image export using the "kaleido" engine requires the Kaleido package,


## Epoch x Round Heatmaps

In [16]:
if tuning_summary.empty:
    print('No tuning summary to plot.')
else:
    heatmap_cases = (
        tuning_summary[['Method key', 'Dataset', 'Model key', 'Setting key']]
        .drop_duplicates()
        .sort_values(['Dataset', 'Model key', 'Setting key', 'Method key'])
    )
    heatmap_selected_pairs = optimal_epoch_round_pairs if 'optimal_epoch_round_pairs' in globals() else tuning_selected
    heatmap_png_warnings = []
    heatmap_saved_png = []
    for _, row in heatmap_cases.iterrows():
        fig = make_tuning_heatmap(
            tuning_summary,
            method=row['Method key'],
            dataset=row['Dataset'],
            model=row['Model key'],
            setting=row['Setting key'],
            selected_pairs=heatmap_selected_pairs,
            plateaus=tuning_plateaus,
            show_markers=True,
            tolerance=1.0,
        )
        if fig is None:
            continue
        output_stem = FIGURE_DIR / f"heatmap_{row['Method key']}_{row['Dataset']}_{row['Model key']}_{row['Setting key']}"
        png_path = _save_plot(fig, output_stem, heatmap_png_warnings)
        if png_path is not None:
            heatmap_saved_png.append(png_path)
        fig.show()
    if heatmap_saved_png:
        print(f'Wrote {len(heatmap_saved_png)} annotated heatmap PNG files to {FIGURE_DIR}.')
    if heatmap_png_warnings:
        print('Heatmap PNG export skipped or incomplete. Install/enable kaleido for static PNG export.')
        print(f'First heatmap PNG export warning: {heatmap_png_warnings[0]}')


Heatmap PNG export skipped or incomplete. Install/enable kaleido for static PNG export.
First heatmap PNG export warning: Image export using the "kaleido" engine requires the Kaleido package,
